In [ ]:
# ==========================================
# 1. INITIALIZATION & DATA LOADING
# ==========================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Set visual style
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = [12, 6]

# Paths - Assuming standard project structure
processed_data_path = r"C:\study\KineticDispatcher\data\processed\optimized_dispatch_results.csv"

if os.path.exists(processed_data_path):
    df = pd.read_csv(processed_data_path)
    print(f"✅ Data Loaded: {len(df)} orders found.")
else:
    print("❌ Data not found! Run the TruthLinkOrchestrator first.")
    # Stop execution or provide dummy logic for structure
    raise FileNotFoundError("Run orchestrator to generate results.")

# ==========================================
# 2. FEATURE ENGINEERING (Metrics Calculation)
# ==========================================
# Baseline: Rider arrives exactly when merchant clicks 'Ready'
df['baseline_wait'] = np.maximum(0, df['actual_handover'] - df['ready_click'])

# TruthLink: Rider arrives at Dispatch Time + Travel ETA
df['rider_arrival'] = df['dispatch_in_mins'] + df['rider_eta']
df['truthlink_wait'] = np.maximum(0, df['actual_handover'] - df['rider_arrival'])

# Error Metrics
df['prediction_error'] = df['t_for_predicted'] - df['actual_handover']
df['cumulative_savings'] = (df['baseline_wait'] - df['truthlink_wait']).cumsum()

# ==========================================
# 3. CHART 1: THE SIGNAL DE-NOISING (Time-Series)
# ==========================================
# Visualizes how Physics & AI lift the untrusted signal to reality.
plt.figure(figsize=(15, 6))
sample = df.head(40).reset_index()
plt.plot(sample.index, sample['actual_handover'], label='Ground Truth (Handover)', color='#2ecc71', marker='o', linewidth=3)
plt.plot(sample.index, sample['ready_click'], label='Noisy Merchant Signal', color='#e74c3c', linestyle='--', alpha=0.6)
plt.plot(sample.index, sample['t_for_predicted'], label='TruthLink Prediction', color='#3498db', linewidth=2)

plt.fill_between(sample.index, sample['ready_click'], sample['actual_handover'], color='red', alpha=0.1, label='Baseline Idle Waste')
plt.title("Visual Analysis: Signal De-noising & Physics Overrides", fontsize=16)
plt.ylabel("Minutes from T=0")
plt.legend()
plt.show()



# ==========================================
# 4. CHART 2: WAIT TIME DISTRIBUTION (KDE)
# ==========================================
# Proves we have shifted the fleet toward zero-wait.
plt.figure(figsize=(10, 5))
sns.kdeplot(df['baseline_wait'], fill=True, color="#e74c3c", label=f"Baseline (Avg: {df['baseline_wait'].mean():.1f}m)")
sns.kdeplot(df['truthlink_wait'], fill=True, color="#2ecc71", label=f"TruthLink (Avg: {df['truthlink_wait'].mean():.1f}m)")
plt.axvline(x=0, color='black', linestyle='--')
plt.title("Impact on Rider Wait Waste (RWW)", fontsize=16)
plt.xlabel("Minutes Spent Idling at Restaurant")
plt.legend()
plt.show()



# ==========================================
# 5. CHART 3: RIDER HOURLY COMPOSITION (ROI)
# ==========================================
# Shows the increase in 'Active Transit' time vs 'Idle' time.
metrics_df = pd.DataFrame({
    'System': ['Old Baseline', 'TruthLink AI'],
    'Transit (Productive)': [df['rider_eta'].mean(), df['rider_eta'].mean()],
    'Wait (Unproductive)': [df['baseline_wait'].mean(), df['truthlink_wait'].mean()]
})

metrics_df.set_index('System').plot(kind='bar', stacked=True, color=['#3498db', '#e74c3c'])
plt.title("Rider Hour Utilization: Productive Transit vs. Idle Wait", fontsize=14)
plt.ylabel("Avg Minutes per Order")
plt.xticks(rotation=0)
plt.show()



# ==========================================
# 6. CHART 4: SYSTEM STABILITY (Rush-Hour Error)
# ==========================================
# Proves $C_{rush}$ maintains accuracy during peak stress.
rush_stats = df.groupby('hour')['prediction_error'].mean()
plt.figure(figsize=(12, 5))
plt.plot(rush_stats.index, rush_stats.values, marker='s', color='#8e44ad', linewidth=2)
plt.axhline(y=0, color='black', linestyle='-')
plt.axhspan(-2, 2, facecolor='#2ecc71', alpha=0.2, label='Safety Tolerance (±2m)')
plt.title("System Robustness: Prediction Bias across 24h Cycle", fontsize=14)
plt.xlabel("Hour of Day (Rush Peaks at 13:00 & 20:00)")
plt.ylabel("Prediction Bias (Mins)")
plt.legend()
plt.show()



# ==========================================
# 7. CHART 5: MERCHANT AUDIT (Reliability Heatmap)
# ==========================================
# Identifies 'Dishonest' merchants for business intervention.
pivot = df.pivot_table(index='merchant_name', values='baseline_wait', aggfunc='mean').sort_values('baseline_wait', ascending=False).head(15)
plt.figure(figsize=(8, 10))
sns.heatmap(pivot, annot=True, cmap="YlOrRd", cbar_klabel="Avg Wait Mins")
plt.title("Merchant Signal Integrity Audit", fontsize=14)
plt.show()



# ==========================================
# 8. FINAL EXECUTIVE SUMMARY
# ==========================================
savings_pct = (1 - (df['truthlink_wait'].mean() / df['baseline_wait'].mean())) * 100
total_mins = (df['baseline_wait'].sum() - df['truthlink_wait'].sum())

print("\n" + "="*40)
print("     TRUTHLINK PERFORMANCE AUDIT")
print("="*40)
print(f"Total Wait Time Saved:     {total_mins:.1f} Minutes")
print(f"System Efficiency Gain:    {savings_pct:.1f}%")
print(f"Mean Wait (Baseline):      {df['baseline_wait'].mean():.2f} min")
print(f"Mean Wait (TruthLink):     {df['truthlink_wait'].mean():.2f} min")
print(f"Signal Integrity Flagging: {len(df[df['baseline_wait'] > 10])} High-Noise Orders Resolved")
print("="*40)

C:\Users\krunal\AppData\Local\Temp\ipykernel_15592\3693062641.py:4: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


❌ Data not found! Run the TruthLinkOrchestrator first.


FileNotFoundError: Run orchestrator to generate results.